In [4]:
import pandas as pd

df = pd.read_csv("data_clean.csv")
df_original = pd.read_csv("data.csv")

In [5]:
df.columns

Index(['Age ', 'BMI', 'Tumor size', 'Tumor size.1', 'Tumor size.2',
       'Last BMI on file', 'Length of followup',
       'Pre-op visual impairment (y/n) - symptomatic complaint of visual loss',
       'Pre-op hydrocephalus (y/n)', 'Pre-op intacranial pressure symptoms',
       'Pre-op headache', 'Pre-op endocrine changes (y/n)',
       'Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional',
       'Intraventricular extension', 'CSF leak', 'Mortality Y/N',
       'Prior operation', 'recurring tumor?',
       'Recurrence after this operation', 'Shunt (y/n) ', 'Transient DI',
       'Permanent DI',
       'DDAVP (y/n) - most patients get periop, can do time frame (in hospital vs 1 mo vs 1 yr postop)',
       'DDAVP outside of hospital', 'DDAVP ≥1 year', 'Post-op new panhypopit',
       'Other vascular complication (need to specify)',
       'Hemorrhage or hemorrhagic stroke', 'Ischemic Stroke',
       'Other intracranial infection', 'Non-intracranial infectio

In [6]:
preop_features = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2",
    "Tumor volume", "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension",
    "Surgical Approach_craniotomy", "Surgical Approach_endoscopic",
    "Tumor (Adamantinomatous, papillary)_Adamantinomatous ",
    "Tumor (Adamantinomatous, papillary)_Papillary",
    "Location (purely intrasellar, suprasellar/sellar, suprasellar/sellar with pseudoventricular extension, suprasellar/sellar with intraventricular extension, purely intraventricular)_Purely intraventricular",
    "Location (purely intrasellar, suprasellar/sellar, suprasellar/sellar with pseudoventricular extension, suprasellar/sellar with intraventricular extension, purely intraventricular)_suprasellar/sellar",
    "Location (purely intrasellar, suprasellar/sellar, suprasellar/sellar with pseudoventricular extension, suprasellar/sellar with intraventricular extension, purely intraventricular)_suprasellar/sellar with pseudoventricular extension",
    "Solid, cystic, mixed_cystic", "Solid, cystic, mixed_mixed", "Solid, cystic, mixed_solid"
]


In [8]:
!pip install --user lazypredict==0.2.16


  Using cached lazypredict-0.2.16-py2.py3-none-any.whl.metadata (13 kB)
  Using cached pytest_runner-6.0.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached mlflow-3.4.0-py3-none-any.whl.metadata (30 kB)
  Using cached mlflow_skinny-3.4.0-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_tracing-3.4.0-py3-none-any.whl.metadata (19 kB)
  Using cached cryptography-45.0.7-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached fastmcp-2.12.4-py3-none-any.whl.metadata (19 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached databricks_sdk-0.67.0-py3-none-any.whl.metadata (39 kB)
  Using cached fastapi-0.117.1-py3-none-any.whl.metadata (28 kB)
  Using cached opentelemetry_api-1.37.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_proto-1.37.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_s

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [9]:
import pandas as pd
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Feature spaces
preop_features = [...]   # from above
intraop_features = [...] # from above

# Define target variables
targets = [
    "CSF leak", "Transient DI", "Permanent DI", "Post-op new panhypopit",
    "Other vascular complication (need to specify)", 
    "Hemorrhage or hemorrhagic stroke", "Ischemic Stroke",
    "Post-op visual improvement/stable/worse (subjective); should add something specific about visual fields if any patients have; add about visual acuity_worse"  # example target
]

results = {}

for target in targets:
    if target not in df.columns:
        continue
    
    # Define X and y
    X = df[preop_features + intraop_features]
    y = df[target]
    
    # Skip if target is all one class
    if len(y.unique()) < 2:
        print(f"Skipping {target} - only one class")
        continue
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # Run LazyPredict
    clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    models, predictions = clf.fit(X_train, X_test, y_train, y_test)
    
    results[target] = models
    print(f"\n==== Results for {target} ====")
    print(models.head(10))  # top 10 models
    print("\nClassification report:")
    print(classification_report(y_test, predictions["RandomForestClassifier"]))


ModuleNotFoundError: No module named 'lazypredict'